## 1. Use the sqlite3 library to create a connection to the database.

In [13]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('../data/checking-logs.sqlite')

## 2. Using one query per group, create two dataframes, test_results and control_results, with the columns "time" and "avg_diff" and two rows.
The "time" column should contain the values "after" and "before".
The "avg_diff" should contain the average delta for all users for the time period before and after their first visit to the page.
Only take into account users with observations before and after.

In [14]:
query = """
SELECT
    time,
    AVG(user_avg_diff) AS avg_diff
FROM (
    SELECT
        t.uid,
        CASE
            WHEN t.first_commit_ts < t.first_view_ts THEN 'before'
            ELSE 'after'
        END AS time,
        AVG(
            (d.deadlines - strftime('%s', t.first_commit_ts)) / 3600.0
        ) AS user_avg_diff
    FROM test AS t
    JOIN deadlines AS d
        ON t.labname = d.labs
    WHERE t.labname != 'project1'
      AND t.uid IN (
          SELECT t2.uid
          FROM test t2
          WHERE t2.labname != 'project1'
          GROUP BY t2.uid
          HAVING SUM(t2.first_commit_ts <  t2.first_view_ts) > 0
             AND SUM(t2.first_commit_ts >= t2.first_view_ts) > 0
      )
    GROUP BY t.uid, time
)
GROUP BY time;
"""


test_results = pd.read_sql(query, conn)
test_results

,time,avg_diff
0,after,100.178181
1,before,66.679583


In [15]:
query = """
SELECT
    time,
    AVG(user_avg_diff) AS avg_diff
FROM (
    SELECT
        c.uid,
        CASE
            WHEN c.first_commit_ts < c.first_view_ts THEN 'before'
            ELSE 'after'
        END AS time,
        AVG(
            (d.deadlines - strftime('%s', c.first_commit_ts)) / 3600.0
        ) AS user_avg_diff
    FROM control AS c
    JOIN deadlines AS d
        ON c.labname = d.labs
    WHERE c.labname != 'project1'
      AND c.uid IN (
          SELECT c2.uid
          FROM control c2
          WHERE c2.labname != 'project1'
          GROUP BY c2.uid
          HAVING SUM(c2.first_commit_ts <  c2.first_view_ts) > 0
             AND SUM(c2.first_commit_ts >= c2.first_view_ts) > 0
      )
    GROUP BY c.uid, time
)
GROUP BY time;
"""


control_results = pd.read_sql(query, conn)
control_results

,time,avg_diff
0,after,118.165599
1,before,176.306267


In [16]:
conn.close()